#### Creating mounting points for **blob and adls storage**

In [0]:
# load service principle secrets from Key Vault
application_id = dbutils.secrets.get(scope="geeks-key", key="application-id")
service_credential_key_name = dbutils.secrets.get(scope="geeks-key", key="service-credential-key-name")
directory_id = dbutils.secrets.get(scope="geeks-key", key="directory-id")

# Load blob secrets from Key Vault
blob_container = dbutils.secrets.get(scope="geeks-key", key="container")
blob_storageaccount = dbutils.secrets.get(scope="geeks-key", key="storageaccount")
mount_blob = '/mnt/geeks/src_blob'

# Load secrets of raw adls from Key Vault
adls_container = dbutils.secrets.get(scope="geeks-key", key="containerADLS")
adls_storageaccount = dbutils.secrets.get(scope="geeks-key", key="storageaccountADLS")
mount_raw = "/mnt/geeks/rw_adls"

# Load secrets cleansed from Key Vault
cld_container = dbutils.secrets.get(scope="geeks-key", key="containerCleansed")
cld_storageaccount = dbutils.secrets.get(scope="geeks-key", key="storageaccountADLS")
mount_cld = "/mnt/geeks/cld_adls"

# Load preconfig details
configs = {"fs.azure.account.auth.type": "OAuth",
          "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
          "fs.azure.account.oauth2.client.id": f"{application_id}",
          "fs.azure.account.oauth2.client.secret": service_credential_key_name,
          "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{directory_id}/oauth2/token"}

# Tuples for each set
blob_info = (mount_blob, blob_container, blob_storageaccount, configs)
adls_info = (mount_raw, adls_container, adls_storageaccount, configs)
cld_info = (mount_cld, cld_container, cld_storageaccount, configs)

# List of all sets
all_mounts = [blob_info, adls_info, cld_info]

In [0]:
# Method to create mounts in databricks
def create_mounts(mount_path, container, storage_account, mount_configs):
    print("Mounting: ", mount_path)
    
    # Unmount if already mounted
    if any(mount.mountPoint == mount_path for mount in dbutils.fs.mounts()):
        dbutils.fs.unmount(mount_path)

    # Attempt to mount
    try:
        dbutils.fs.mount(
            source = f"abfss://{container}@{storage_account}.dfs.core.windows.net/",
            mount_point = mount_path,
            extra_configs = mount_configs)
        
        print(f"created mount: '{mount_path}' successfully for container: '{container}' in storage_account: '{storage_account}'")
    except Exception as e:
        print(f"Not created mount:'{mount_path}' for container: '{container}' in storage_account: '{storage_account}'\n", e)


In [0]:
print("Started creating all mount points")
for mnt, conatiner, storage, conf in all_mounts:
    create_mounts(mnt, conatiner, storage, conf)
print("completed mount points")

In [0]:
display(dbutils.fs.mounts())


In [0]:
display(dbutils.fs.ls("/mnt/geeks/src_blob/"))